# Stage 1 — Predict Decline

**What this notebook does:** estimates how much each of the 5 cognitive domains is likely to **worsen over the next year** for a patient. That "will-worsen" signal becomes **one input to the SAW recommender** in Stage 2 (which combines *weak now* + *will worsen* to choose what to train).

**Why a separate notebook:** Stage 0 made *one row per person* (a snapshot). Decline is **change between visits**, so here we use the **longitudinal** data — every yearly visit — and reuse the saved healthy ruler (`models/zscore_reference.json`) so scores are on the exact same scale.

**Plan:** 1) build the decline dataset (score every visit) → 2) build the decline signal → 3) validate → 4) save for SAW.

## Setup

Load the tools, the file path, the 5-domain definitions, the valid score ranges, and the **saved z-ruler** from Stage 0. Loading the same ruler is the "same-scale contract": a patient's scores must be measured against the identical healthy baseline the project trained on.

In [ ]:
import os, json
import numpy as np, pandas as pd

PATH = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system\data\investigator_ftldlbd_nacc74.csv"
PROJ = os.path.dirname(os.path.dirname(PATH))            # .../cognitive-train-system
REF_PATH = os.path.join(PROJ, "models", "zscore_reference.json")

ID, VISIT, VER, TIME = "NACCID", "NACCVNUM", "FORMVER", "NACCFDYS"   # NACCFDYS = days since first visit

NEURO = {
    "memory":       ["CRAFTVRS", "CRAFTURS", "CRAFTDVR", "CRAFTDRE", "UDSBENTD"],
    "attention":    ["DIGFORCT", "DIGFORSL", "DIGBACCT", "DIGBACLS", "TRAILA"],
    "executive":    ["TRAILB", "UDSVERFC", "UDSVERLC", "UDSVERTN"],
    "language":     ["MINTTOTS", "ANIMALS", "VEG"],
    "visuospatial": ["UDSBENTC"],
}
NEURO_COLS = sum(NEURO.values(), [])

RANGES = {
    "CRAFTVRS": (0, 44), "CRAFTURS": (0, 25), "CRAFTDVR": (0, 44), "CRAFTDRE": (0, 25),
    "UDSBENTD": (0, 17), "UDSBENTC": (0, 17),
    "DIGFORCT": (0, 14), "DIGFORSL": (0, 9), "DIGBACCT": (0, 14), "DIGBACLS": (0, 8),
    "TRAILA": (0, 150), "TRAILB": (0, 300),
    "UDSVERFC": (0, 40), "UDSVERLC": (0, 40), "UDSVERTN": (0, 80),
    "MINTTOTS": (0, 32), "ANIMALS": (0, 77), "VEG": (0, 77),
}

reference = json.load(open(REF_PATH))
print("Loaded z-ruler for", len(reference), "tests")
print("From:", REF_PATH)

## Step 1 — Build the decline dataset (score every visit)

To see decline we need each person's **whole history**, not one snapshot. This cell:

1. Loads the columns we need for **all** visits.
2. Keeps UDS **v3 + v4** and recodes "not done" codes to blanks (same rules as Stage 0).
3. Rebuilds the **same two-arm cohort** (AD / LBD, drop mixed-AD) — the counts should match Stage 0 (AD 9,882 / LBD 1,084).
4. Converts every visit's test scores to **z-scores** with the saved ruler (flipping the timed Trails), then averages them into the **5 domain scores per visit**.
5. Saves a tidy **longitudinal table** — one row per visit: `NACCID, visit, days, age, 5 domains, diagnosis` — to `data/` (gitignored).

The result (`long`) is the "photo album" the decline signal will learn from. We also report how many people have **≥ 2 scored visits**, since you need at least two points to measure change.

In [ ]:
# 1) load the columns we need, for ALL visits (longitudinal)
cols = [ID, VISIT, VER, TIME, "NACCAGE", "NACCETPR", "CVDIF"] + NEURO_COLS
raw = pd.read_csv(PATH, usecols=cols, low_memory=False)
for c in raw.columns:
    if c != ID:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

# 2) keep UDS v3+v4, recode "not done" codes -> NaN
v = raw[VER].astype(str).str.strip()
uds = raw[v.str.startswith("3") | v.str.startswith("4")].copy()
for col in NEURO_COLS:
    lo, hi = RANGES[col]
    uds[col] = uds[col].where(uds[col].between(lo, hi))

# 3) two-arm cohort: most-recent etiology; drop mixed-AD, keep mixed-LBD
last = uds.sort_values(VISIT).groupby(ID)["NACCETPR"].last()
AD_ids, LBD_ids = last.index[last == 1], last.index[last == 2]
vasc_ever = uds.groupby(ID)["CVDIF"].apply(lambda s: s.isin([1, 2]).any())
AD_clean  = AD_ids[~vasc_ever.reindex(AD_ids).fillna(False).values]
keep_ids  = AD_clean.union(LBD_ids)
diagnosis = pd.Series(index=keep_ids, dtype=object)
diagnosis.loc[AD_clean] = "AD"
diagnosis.loc[LBD_ids]  = "LBD"

# 4) score EVERY cohort visit with the saved ruler (flip the timed Trails)
coh = uds[uds[ID].isin(keep_ids)].copy()
for col, r in reference.items():
    if col in coh.columns and r["sd"] > 0:
        z = (coh[col] - r["mean"]) / r["sd"]
        coh[col] = -z if not r["higher_is_better"] else z

# 5) one row per visit: id, visit, days-since-first-visit, age, 5 domain z-scores, diagnosis
long = pd.DataFrame({ID: coh[ID].values, "visit": coh[VISIT].values,
                     "days": coh[TIME].values, "age": coh["NACCAGE"].values})
for d, cs in NEURO.items():
    use = [c for c in cs if c in coh.columns]
    long[d] = coh[use].mean(axis=1, skipna=True).values
long["diagnosis"] = long[ID].map(diagnosis)

OUT = os.path.join(PROJ, "data", "longitudinal_scores.parquet")
long.to_parquet(OUT, index=False)
print("Saved:", OUT)

# summary
dom = list(NEURO)
vpp = long.dropna(subset=dom, how="all").groupby(ID).size()   # scored visits per person
print(f"\nCohort: {long[ID].nunique():,} people | {len(long):,} visit-rows")
print(f"Diagnosis (people): {diagnosis.value_counts().to_dict()}")
print(f"Scored visits per person: min {vpp.min()}, median {int(vpp.median())}, max {vpp.max()}")
print(f"People with >=2 scored visits (usable for decline): {(vpp >= 2).sum():,}")
print("\nPreview (one person's history):")
print(long.sort_values([ID, 'visit']).head(8).round(2).to_string(index=False))